In [ ]:
""" import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make plots a little larger
plt.rcParams["figure.figsize"] = (14, 6) """


In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

while not os.path.isdir("src") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

sys.path.insert(0, "src")


from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir("data/egx")

print(feed.symbols)

In [ ]:
print("Number of assets:", feed.n_assets)
print("Number of trading days:", feed.n_days)

In [ ]:
print(type(feed.close))
print(type(feed.volume))

In [ ]:
print(feed.close[:5])

In [ ]:
print(feed.symbols[:5])

In [ ]:
abuk = feed.close[:, 0]
print(abuk)

In [ ]:
asset = "ABUK"

# Find the column index
asset_idx = feed.symbols.index(asset)

# Extract the closing prices
close = feed.close[:, asset_idx]

print(close[:10])

In [ ]:
# Create a DataFrame for the selected asset

df = pd.DataFrame({
    "Date": pd.to_datetime(feed.dates),
    "Close": close
})

df.set_index("Date", inplace=True)

df.head()

In [ ]:
print(df.head())

print(df.tail())

print(df.info())

print(df.describe())

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(df.index, df["Close"])

plt.title(f"{asset} Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")

plt.grid(True)

plt.show()

In [ ]:
# =====================================================
# 3. Feature Engineering
# =====================================================

# Calculate Moving Averages
df["MA9"] = df["Close"].rolling(window=9).mean()

df["MA20"] = df["Close"].rolling(window=20).mean()

df.head(25)

In [ ]:
df[["Close", "MA9", "MA20"]].head(25)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df.index, df["Close"], label="Close", linewidth=1)

plt.plot(df.index, df["MA9"], label="MA9", linewidth=2)

plt.plot(df.index, df["MA20"], label="MA20", linewidth=2)

plt.title(f"{asset} - Moving Average Strategy")

plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# =====================================================
# 4. Backtesting
# =====================================================

initial_cash = 1000.0

cash = initial_cash
shares = 0.0

position = 0

portfolio_values = []

buy_dates = []
buy_prices = []

sell_dates = []
sell_prices = []

In [ ]:
# =====================================================
# Walk-Forward Backtest
# =====================================================

for date, row in df.iterrows():

    price = row["Close"]

    # Skip until MA20 is available
    if pd.isna(row["MA20"]):
        portfolio_values.append(cash)
        continue

    # =====================
    # BUY
    # =====================
    if row["MA9"] > row["MA20"] and position == 0:

        shares = cash / price
        cash = 0

        position = 1

        buy_dates.append(date)
        buy_prices.append(price)

    # =====================
    # SELL
    # =====================
    elif row["MA9"] < row["MA20"] and position == 1:

        cash = shares * price
        shares = 0

        position = 0

        sell_dates.append(date)
        sell_prices.append(price)

    # Portfolio value today
    portfolio = cash + shares * price

    portfolio_values.append(portfolio)

In [ ]:
print("Buy operations :", len(buy_dates))
print("Sell operations:", len(sell_dates))

In [ ]:
df["Portfolio"] = portfolio_values

df.tail()

In [ ]:
""" abuk = feed.close[:, 0]
print(abuk) """

In [ ]:
final_portfolio_value = portfolio_values[-1]

print(f"Initial Portfolio Value : {initial_cash:.2f} EGP")
print(f"Final Portfolio Value   : {final_portfolio_value:.2f} EGP")

In [ ]:
total_return = (final_portfolio_value - initial_cash) / initial_cash * 100
return_per_year = total_return / (len(df) / 252) #assuming 252 tradining days in a year

print(f"Total Return : {total_return:.2f}%")
print(f"return per year: {return_per_year:.2f}%")

In [ ]:
portfolio_series = pd.Series(portfolio_values, index=df.index)

running_max = portfolio_series.cummax()

drawdown = (portfolio_series - running_max) / running_max

max_drawdown = drawdown.min()

print(f"Maximum Drawdown: {max_drawdown:.2%}")

In [ ]:
print("=" * 40)
print("Moving Average Strategy Results")
print("=" * 40)

print(f"Initial Capital      : {initial_cash:.2f} EGP")
print(f"Final Portfolio      : {final_portfolio_value:.2f} EGP")
print(f"Total Return         : {total_return:.2f}%")
print(f"Maximum Drawdown     : {max_drawdown:.2%}")
print(f"Buy Operations       : {len(buy_dates)}")
print(f"Sell Operations      : {len(sell_dates)}")

In [ ]:
plt.figure(figsize=(16,8))

plt.plot(df.index, df["Close"], label="Close", color="black", alpha=0.7)

plt.plot(df.index, df["MA9"], label="MA9", linewidth=2)

plt.plot(df.index, df["MA20"], label="MA20", linewidth=2)

plt.scatter(
    buy_dates,
    buy_prices,
    marker="^",
    color="green",
    s=120,
    label="Buy"
)

plt.scatter(
    sell_dates,
    sell_prices,
    marker="v",
    color="red",
    s=120,
    label="Sell"
)

plt.title(f"{asset} Moving Average Crossover Strategy")

plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(df.index, portfolio_values, color="purple", linewidth=2)

plt.title("Portfolio Value Over Time")

plt.xlabel("Date")

plt.ylabel("Portfolio Value (EGP)")

plt.grid(True)

plt.show()

In [ ]:
print(df.index.min())
print(df.index.max())

In [ ]:
print(feed.dates[:10])